In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

ARTIFACTS_PUBLIC = PROJECT_ROOT / "artifacts" / "public"
DATA_PUBLIC = PROJECT_ROOT / "data" / "public"
MODELS_PUBLIC = PROJECT_ROOT / "models" / "public"

# Day03 · Backtest postinferencia por albarán

Notebook de evidencia para la política `PRODUCT_002_follow_PRODUCT_003_SUPPLIER_009` con métricas before/after persistidas.

In [2]:
from pathlib import Path
import sys

# Resolver raíz del repo tanto si ejecutas desde /repo como desde /repo/notebooks
CWD = Path.cwd().resolve()
if (CWD / "artifacts" / "public").exists():
    PROJECT_ROOT = CWD
elif (CWD.parent / "artifacts" / "public").exists():
    PROJECT_ROOT = CWD.parent
else:
    raise FileNotFoundError(f"No encuentro /artifacts/public ni en {CWD} ni en {CWD.parent}")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

REPORTS_ROOT = PROJECT_ROOT / "artifacts" / "public"
CANDIDATES_ROOT = REPORTS_ROOT / "metrics" / "candidates"
BASELINE_ROOT = REPORTS_ROOT / "metrics" / "baseline"

assert CANDIDATES_ROOT.exists(), f"No existe carpeta candidates: {CANDIDATES_ROOT}"
assert BASELINE_ROOT.exists(), f"No existe carpeta baseline: {BASELINE_ROOT}"

# Este notebook documenta el backtest Day03 y debe apuntar al run summary específico de 20260305.
CANDIDATES_DIR = CANDIDATES_ROOT / "20260305"
assert CANDIDATES_DIR.exists(), f"No existe carpeta Day03 esperada: {CANDIDATES_DIR}"
RUN_DATE = CANDIDATES_DIR.name

baseline_files = sorted(BASELINE_ROOT.glob("*_postinferencia_albaran_baseline_metrics.json"))
assert baseline_files, "No hay baseline Day03 postinferencia."
BASELINE_PATH = baseline_files[-1]

patterns = [
    "*_BASELINE_WITH_DETERMINISTIC_LAYER_PRODUCT_002_FOLLOW_PRODUCT_003_SUPPLIER_009_v1_run_summary.json",
    "*_POLICY_PRODUCT_002_FOLLOW_PRODUCT_003_SUPPLIER_009_v1_run_summary.json",
    "*_run_summary.json",
]

run_summary_files = []
for pat in patterns:
    run_summary_files = sorted(CANDIDATES_DIR.glob(pat))
    if run_summary_files:
        break

assert run_summary_files, f"No hay run_summary en {CANDIDATES_DIR}"
RUN_SUMMARY_PATH = run_summary_files[-1]

print("PROJECT_ROOT:", ".")
print("RUN_DATE:", RUN_DATE)
print("BASELINE_PATH:", BASELINE_PATH.relative_to(PROJECT_ROOT))
print("RUN_SUMMARY_PATH:", RUN_SUMMARY_PATH.relative_to(PROJECT_ROOT))
RUN_SUMMARY_PATH.relative_to(PROJECT_ROOT)

PROJECT_ROOT: .
RUN_DATE: 20260305
BASELINE_PATH: artifacts/public/metrics/baseline/20260305_postinferencia_albaran_baseline_metrics.json
RUN_SUMMARY_PATH: artifacts/public/metrics/candidates/20260305/20260305T160207Z_BASELINE_WITH_DETERMINISTIC_LAYER_PRODUCT_002_FOLLOW_PRODUCT_003_SUPPLIER_009_v1_run_summary.json


PosixPath('artifacts/public/metrics/candidates/20260305/20260305T160207Z_BASELINE_WITH_DETERMINISTIC_LAYER_PRODUCT_002_FOLLOW_PRODUCT_003_SUPPLIER_009_v1_run_summary.json')

## Carga de artefactos persistidos

Se leen baseline/candidate y auditorías (`detalle`, `resumen_evento`, `resumen_albaran`) desde la ejecución más reciente del día.

In [3]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

baseline_payload = json.loads(BASELINE_PATH.read_text(encoding="utf-8"))
run_summary_payload = json.loads(RUN_SUMMARY_PATH.read_text(encoding="utf-8"))

candidate_metrics_path = PROJECT_ROOT / Path(run_summary_payload["candidate_metrics_json"])
candidate_payload = json.loads(candidate_metrics_path.read_text(encoding="utf-8"))

audit_paths = {key: PROJECT_ROOT / Path(value) for key, value in run_summary_payload["audit_paths"].items()}
df_detail = pd.read_csv(audit_paths["detail"])
df_resumen_evento = pd.read_csv(audit_paths["resumen_evento"])
df_resumen_albaran = pd.read_csv(audit_paths["resumen_albaran"])

display(pd.DataFrame([baseline_payload["metrics"]]))
display(pd.DataFrame([candidate_payload["metrics"]]))
display(df_resumen_albaran.head(20))

,accuracy,balanced_accuracy,f1_pos,top1_hit,top2_hit,coverage,test_events,pair_groups_PRODUCT_002_PRODUCT_003,coherence_before,coherence_after,coherence_delta,overrides_count,overrides_improved,overrides_harmed,overrides_neutral
0,0.909505,0.861463,0.689893,0.772123,0.862894,1.0,2633,365,0.915068,1.0,0.084932,44,44,0,2589


,accuracy,balanced_accuracy,f1_pos,top1_hit,top2_hit,coverage,test_events,top1_hit_before,top2_hit_before,top1_hit_after,...,pair_groups_PRODUCT_002_PRODUCT_003,coherence_before,coherence_after,coherence_delta,overrides_count,overrides_improved,overrides_harmed,overrides_neutral,coverage_before,coverage_after
0,0.909505,0.861463,0.689893,0.788834,0.862894,1.0,2633,0.772123,0.862894,0.788834,...,365,0.915068,1.0,0.084932,44,44,0,2589,1.0,1.0


,fecha_evento,albaran_id,eventos_total,eventos_override,coherence_before,coherence_after,policy_applied,policy_reason,run_id,ts_utc
0,2028-02-22,AE 179,1,0,NaN,NaN,0,NaN,20260305T160209Z,2026-03-05T16:02:09.052211+00:00
1,2028-02-22,AE 180,4,0,NaN,NaN,0,NaN,20260305T160209Z,2026-03-05T16:02:09.052211+00:00
2,2028-02-22,AE 181,4,0,1.0,1.0,0,NaN,20260305T160209Z,2026-03-05T16:02:09.052211+00:00
3,2028-02-22,AE 182,4,0,1.0,1.0,0,NaN,20260305T160209Z,2026-03-05T16:02:09.052211+00:00
4,2028-02-27,AE 196,4,0,NaN,NaN,0,NaN,20260305T160209Z,2026-03-05T16:02:09.052211+00:00
5,2028-02-27,AE 197,4,1,0.0,1.0,1,PRODUCT_002_follow_PRODUCT_003_SUPPLIER_009,20260305T160209Z,2026-03-05T16:02:09.052211+00:00
6,2028-02-27,AE 198,4,1,0.0,1.0,1,PRODUCT_002_follow_PRODUCT_003_SUPPLIER_009,20260305T160209Z,2026-03-05T16:02:09.052211+00:00
7,2028-02-27,AE 199,4,0,NaN,NaN,0,NaN,20260305T160209Z,2026-03-05T16:02:09.052211+00:00
8,2028-02-27,AE 200,4,0,NaN,NaN,0,NaN,20260305T160209Z,2026-03-05T16:02:09.052211+00:00
9,2028-02-27,AE 201,4,0,NaN,NaN,0,NaN,20260305T160209Z,2026-03-05T16:02:09.052211+00:00


## Comparativa before/after y coherencia

Validación mínima: la política no debe reducir coherencia PRODUCT_002/PRODUCT_003 en grano `fecha_evento + albaran_id`.

In [4]:
m = candidate_payload["metrics"]
comparison = pd.DataFrame([
    {"metric": "top1_hit_before", "value": m.get("top1_hit_before")},
    {"metric": "top1_hit_after", "value": m.get("top1_hit_after")},
    {"metric": "top2_hit_before", "value": m.get("top2_hit_before")},
    {"metric": "top2_hit_after", "value": m.get("top2_hit_after")},
    {"metric": "coherence_before", "value": m.get("coherence_before")},
    {"metric": "coherence_after", "value": m.get("coherence_after")},
    {"metric": "coherence_delta", "value": m.get("coherence_delta")},
    {"metric": "overrides_count", "value": m.get("overrides_count")},
    {"metric": "overrides_improved", "value": m.get("overrides_improved")},
    {"metric": "overrides_harmed", "value": m.get("overrides_harmed")},
    {"metric": "overrides_neutral", "value": m.get("overrides_neutral")},
])
display(comparison)

if m.get("coherence_before") is not None and m.get("coherence_after") is not None:
    assert float(m["coherence_after"]) >= float(m["coherence_before"]), "La coherencia after empeora respecto a before."

df_decisions = df_resumen_evento[[
    c for c in ["event_id", "fecha_evento", "albaran_id", "producto_canonico", "decision_pre_policy", "decision_final", "policy_applied_event", "policy_reason_event"]
    if c in df_resumen_evento.columns
]].copy()
display(df_decisions.head(50))

,metric,value
0,top1_hit_before,0.772123
1,top1_hit_after,0.788834
2,top2_hit_before,0.862894
3,top2_hit_after,0.862894
4,coherence_before,0.915068
5,coherence_after,1.000000
6,coherence_delta,0.084932
7,overrides_count,44.000000
8,overrides_improved,44.000000
9,overrides_harmed,0.000000


,event_id,fecha_evento,albaran_id,producto_canonico,decision_pre_policy,decision_final,policy_applied_event,policy_reason_event
0,000a484e296598f8254cbc5b17f9a2d52aafd5bb,2028-06-11,AE 525,PRODUCT_003,SUPPLIER_009,SUPPLIER_009,0,NaN
1,0035d8fca197f917d3e48430d1b254d2b803fb5e,2028-03-13,AE 248,PRODUCT_001,SUPPLIER_007,SUPPLIER_007,0,NaN
2,0061169fdbacefbf48cbec6f5e58286a859a35e5,2029-07-01,AE 487,PRODUCT_002,SUPPLIER_009,SUPPLIER_009,0,NaN
3,006dd3a0945d076432ef962a13a6752fa65143a3,2029-01-10,AE 19,PRODUCT_003,SUPPLIER_009,SUPPLIER_009,0,NaN
4,008dc37fae14ecb378bd681747e145bbbf021910,2029-02-14,AE 104,PRODUCT_003,SUPPLIER_009,SUPPLIER_009,0,NaN
5,00a8f81f214a06264c724a47a2a76422952cc557,2029-06-05,AE 387,PRODUCT_003,SUPPLIER_009,SUPPLIER_009,0,NaN
6,00b573adc401ae51729cc91c18f62f1e00cd6ede,2029-06-25,AE 466,PRODUCT_003,SUPPLIER_009,SUPPLIER_009,0,NaN
7,010e1f1f4222a19b608ae46cb4930ac813b54baa,2029-06-03,AE 379,PRODUCT_003,SUPPLIER_009,SUPPLIER_009,0,NaN
8,011405d096dbdddb7c130fe717526312f48b553e,2028-05-06,AE 408,PRODUCT_003,SUPPLIER_009,SUPPLIER_009,0,NaN
9,0115b8672424edddae7a7cfe57465cfbd900bb48,2029-02-14,AE 102,PRODUCT_003,SUPPLIER_009,SUPPLIER_009,0,NaN


## Cierre

- Este notebook consume artefactos persistidos por `src/ml/rules/scripts/day03_postinference_albaran_backtest.py`.
- La recomendación de promoción se basa en `coherence_delta`, balance de overrides y no regresión de Top-2.